In [33]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [84]:
# print dataframe
df = pd.read_csv("ModeChoiceOptima.txt",sep="\t")
#df.head()
#df.columns
df

,ID,DestAct,NbTransf,TimePT,WalkingTimePT,WaitingTimePT,CostPT,CostCar,TimeCar,NbHousehold,...,FreqTripHouseh,Region,distance_km,Choice,InVehicleTime,ModeToSchool,ReportedDuration,CoderegionCAR,age,Weight
0,10350017,2,4,85,23,10,12.4,3.17,32,2,...,4,1,30.0,1,52,3,255,1,27,0.000379
1,10350020,1,4,108,26,16,12.4,3.28,30,2,...,4,1,32.0,-1,66,3,150,1,28,0.000341
2,10350025,11,2,82,33,5,3.0,0.45,6,-1,...,2,1,4.5,0,44,-1,20,1,-1,0.000368
3,10350075,1,3,107,21,31,24.0,2.36,23,2,...,1,1,25.0,1,55,-1,30,1,63,0.000368
4,10350085,1,5,190,116,18,10.8,1.16,14,3,...,3,1,12.5,1,56,-1,20,1,57,0.000409
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2260,96040535,1,6,299,84,94,48.8,10.72,86,1,...,1,7,119.0,-1,121,6,90,7,51,0.000409
2261,96040537,8,0,139,116,0,14.4,3.18,43,5,...,3,7,32.0,1,23,5,70,7,46,0.000368
2262,96040537,8,0,71,57,0,6.0,0.94,12,5,...,3,7,9.0,1,14,5,20,7,46,0.000368
2263,96040538,11,2,118,70,10,11.4,1.77,24,5,...,4,7,17.5,1,38,3,30,7,49,0.000409


In [112]:
# Listing the critical columns to answer the research questions
# Outcome we want to predict
target = ['Choice']

# Socio-demographic variables (Sub-question 1)
#socio_demo = ['age', 'Gender', 'Education', 'BirthYear', 'LangCode', 'FamilSitu', 'NbHousehold', 'NbChild']
#### not adding age, because maybe this can be calculated with BirthYear
socio_demo = ['Gender', 'Education', 'BirthYear', 'LangCode', 'FamilSitu', 'NbHousehold', 'NbChild']

# Vehicle Ownership & Availability (Sub-question 4)
vehicle = ['NbCar', 'NbMoto', 'CarAvail']

# Spatial context (Sub-question 3)
spatial_context = ['UrbRur', 'TypeCommune', 'ClassifCodeLine', 'Region', 'frequency']

# Trip utility (standard predictors for mode choice)
trip_utility = ['TimePT', 'TimeCar', 'InVehicleTime', 'WalkingTimePT', 'WaitingTimePT', 
                'distance_km', 'NbTransf', 'MarginalCostPT', 'CostCar']

# Historical & Childhood Background
history = ['TripPurpose', 'DestAct', 'NbTrajects', 'FreqTripHouseh', 
                      'ModeToSchool', 'ResidChild', 'FreqTrainPar', 'FreqOtherPar']

# Attitudinal Statements (Sub-question 2: Environmental and Lifestyle)
attitudes = ['Envir01', 'Envir05', 'Mobil01', 'Mobil04', 'Mobil08', 'Mobil13', 
             'Mobil15', 'Mobil23', 'ResidCh05', 'ResidCh07', 'LifSty02', 
             'LifSty06', 'LifSty08', 'LifSty11']

critical_cols = target + socio_demo + vehicle + spatial_context + trip_utility + history

In [113]:
# Datacleaning 
# Replace missing value codes (-1 = missing data) 
df.replace([-1], np.nan, inplace=True) 
# Delete rows with NaN
#df_clean = df.dropna()
#### deleting all columns with NaN causes going from 2265 rows to 886 rows

#### by deleting the missing values, we also do not have "no opinion people"
#df1 = df[df["No_Opinion_Group"] == 1] # 0 = no opinion, but now 1 bc then it is better visualised
#df1

# Delete rows where values in critical columns are missing, but keep the rows where attitudes might be missing for now
df_clean = df.dropna(subset=critical_cols)

# Version where you also drop rows with missing attitudes
df_attitudes_only = df_clean.dropna(subset=attitudes)

# Print results to see how much data remains
print(f"Original rows: {len(df)}")
print(f"Rows after cleaning critical columns: {len(df_clean)}")
print(f"Rows after also cleaning attitude columns: {len(df_attitudes_only)}")

Original rows: 2265
Rows after cleaning critical columns: 1171
Rows after also cleaning attitude columns: 1134


In [ ]:
# Making a no opinion group 
#### this should go before the df1 part
# If any of the answers are -2, the person belongs to the no opinion group
# List the opinion columns from table 2
cols_opinion_table2 = ['FreqTripHouseh', 'ModeToSchool', 'ResidChild', 'FreqCarPar', 'FreqTrainPar', 'FreqOtherPar'] # "FreqOthPar" is noted in the PDF, but it is "FreqOtherPar"

# List attitudal statements
attitudal_statements = [
    'Envir01', 'Envir02', 'Envir03', 'Envir04', 'Envir05', 'Envir06', 
    'Mobil01', 'Mobil02', 'Mobil03', 'Mobil04', 'Mobil05', 'Mobil06',
    'Mobil07', 'Mobil08', 'Mobil09', 'Mobil10', 'Mobil11', 'Mobil12',
    'Mobil13', 'Mobil14', 'Mobil15', 'Mobil16', 'Mobil17', 'Mobil18',
    'Mobil19', 'Mobil20', 'Mobil21', 'Mobil22', 'Mobil23', 'Mobil24',
    'Mobil25', 'Mobil26', 'Mobil27', 'ResidCh01', 'ResidCh02', 'ResidCh03',
    'ResidCh04', 'ResidCh05', 'ResidCh06', 'ResidCh07', 'LifSty01', 'LifSty02',
    'LifSty03', 'LifSty04', 'LifSty05', 'LifSty06', 'LifSty07', 'LifSty08',
    'LifSty09', 'LifSty10', 'LifSty11', 'LifSty12', 'LifSty13', 'LifSty14'
]

opinion_cols = cols_opinion_table2 + attitudal_statements

# Create a new column: 1 if they have 'No Opinion', 0 if they do
df['No_Opinion_Group'] = df[opinion_cols].eq(-2).any(axis=1).astype(int)

# Subset of people who actually provided attitudes
df_attitude = df[df['No_Opinion_Group'] == 0].copy()

# Subset of people who skipped the opinion questions
df_no_opinion = df[df['No_Opinion_Group'] == 1].copy()

# Now check how many people are in each group
print(df['No_Opinion_Group'].value_counts())

No_Opinion_Group
0    2221
1      44
Name: count, dtype: int64


In [114]:
df.head(10)

,ID,DestAct,NbTransf,TimePT,WalkingTimePT,WaitingTimePT,CostPT,CostCar,TimeCar,NbHousehold,...,FreqTripHouseh,Region,distance_km,Choice,InVehicleTime,ModeToSchool,ReportedDuration,CoderegionCAR,age,Weight
0,10350017,2.0,4,85,23,10,12.4,3.17,32,2.0,...,4.0,1,30.0,1.0,52.0,3.0,255.0,1,27.0,0.000379
1,10350020,1.0,4,108,26,16,12.4,3.28,30,2.0,...,4.0,1,32.0,NaN,66.0,3.0,150.0,1,28.0,0.000341
2,10350025,11.0,2,82,33,5,3.0,0.45,6,NaN,...,2.0,1,4.5,0.0,44.0,NaN,20.0,1,NaN,0.000368
3,10350075,1.0,3,107,21,31,24.0,2.36,23,2.0,...,1.0,1,25.0,1.0,55.0,NaN,30.0,1,63.0,0.000368
4,10350085,1.0,5,190,116,18,10.8,1.16,14,3.0,...,3.0,1,12.5,1.0,56.0,NaN,20.0,1,57.0,0.000409
5,10350086,1.0,4,116,38,29,9.6,1.89,20,3.0,...,3.0,1,19.0,1.0,49.0,NaN,30.0,1,58.0,0.000368
6,10350100,4.0,1,20,1,5,3.0,1.14,9,1.0,...,1.0,1,13.0,1.0,14.0,4.0,10.0,1,80.0,0.000202
7,10350101,1.0,4,153,56,19,18.6,4.05,41,4.0,...,3.0,1,35.7,NaN,78.0,4.0,78.0,1,47.0,0.000409
8,10350104,7.0,5,165,67,18,22.2,4.26,40,2.0,...,2.0,1,43.0,NaN,80.0,4.0,110.0,1,78.0,0.000321
9,10350119,9.0,2,88,34,6,12.4,2.92,30,2.0,...,3.0,1,29.0,NaN,48.0,5.0,95.0,1,64.0,0.000098


In [115]:
df_clean.head(10)

,ID,DestAct,NbTransf,TimePT,WalkingTimePT,WaitingTimePT,CostPT,CostCar,TimeCar,NbHousehold,...,FreqTripHouseh,Region,distance_km,Choice,InVehicleTime,ModeToSchool,ReportedDuration,CoderegionCAR,age,Weight
0,10350017,2.0,4,85,23,10,12.4,3.17,32,2.0,...,4.0,1,30.0,1.0,52.0,3.0,255.0,1,27.0,0.000379
10,10350120,4.0,4,116,26,34,24.8,4.87,33,2.0,...,3.0,1,45.0,1.0,56.0,5.0,30.0,1,64.0,0.000098
11,10350125,4.0,1,35,15,4,4.8,0.69,9,1.0,...,3.0,1,7.0,1.0,16.0,4.0,15.0,1,68.0,0.000289
12,10350125,11.0,0,11,10,0,2.2,0.13,1,1.0,...,3.0,1,1.0,1.0,1.0,4.0,7.0,1,68.0,0.000289
14,10350197,1.0,5,124,36,24,11.0,2.44,22,2.0,...,3.0,1,26.0,1.0,64.0,1.0,24.0,1,30.0,0.000379
15,10350199,4.0,4,123,21,59,9.4,2.03,19,3.0,...,2.0,1,22.0,1.0,43.0,5.0,20.0,1,33.0,0.000232
16,10350199,4.0,2,80,26,3,4.8,1.13,12,3.0,...,2.0,1,11.0,1.0,51.0,5.0,20.0,1,33.0,0.000232
20,10350272,1.0,3,266,67,95,37.2,5.34,51,2.0,...,3.0,1,55.0,1.0,104.0,3.0,NaN,1,42.0,0.000409
21,10350272,7.0,5,250,18,137,11.0,5.16,40,2.0,...,3.0,1,49.5,1.0,95.0,3.0,20.0,1,42.0,0.000409
24,10360009,4.0,1,29,12,0,6.2,0.92,9,4.0,...,3.0,1,9.0,1.0,17.0,4.0,10.0,1,69.0,0.000321


In [ ]:
# testing the birthyear/age calculation
# Displays the first 5 rows of just these two columns
#print(df[['BirthYear', 'age']].head(100))

# Check NaN count for just your specific columns
print(df[['BirthYear', 'age']].isnull().sum())

#### conclusion, this is the same. There is no row with "BirthYear" and no "age" and the other way around

BirthYear    147
age          147
dtype: int64


I don't know the "goal" of the variables yet, so I don't know if I should already create extra columns such as using the travel time and waiting time for something